# Reinforcement learning
Beim Reinforcement Learning werden im Gegensatz zu klassischen Machine Learning Methoden keine vorgegebenen Daten verwendet, sondern das Datenset wird während des Prozess generiert und direkt verarbeitet.

Umgebung (Environment)
- Erzeugt die Trainingsdaten indem sie das unterliegende Programm kontinuierlich ausliest und Inputs an das Programm weiter gibt
 -Effektiv das Programm, dass trainiert werden soll

Prozess
 - Schleife die das Programm in Schritten ausführt
 - Inputs vom Agenten nehmen und weitergeben

Agent
 - Kennt den Lernprozess
 - Agent generiert Inputs (Actions) und gibt diese an den Prozess weiter
 - Daraus lernt der Agent positive und negative Actions anhand der generierten Daten
 - Beim Lernen bekommt der Agent für positive Actions Punkte als Belohnung, die der Machine Learning Algorithmus als "Fortschritt" ansieht

## Packeges
- Gym(nasium)
    Stellt Enviroment, Agent, Space, .. zur Verfügung
- Stable Baseline 3
    Stellt einige nützliche Klassen und Funktionen zur Verfügung

In [5]:
import gymnasium as gym
from opt_einsum.backends.object_arrays import object_einsum

## Vordefinierte Umgebung
Es gibt einige von OpenAI vorgegebene Testumgebungen, zum testen verschiedener Methoden

z.B CartPole

Bei dieser geht es darum, ein Kart mit einer Stange möglichst gerade zu halten durch Links- und Rechtsbewegungen

https://github.com/openai/gym/blob/master/gym/envs/classic_control/cartpole.py

In [6]:
env = gym.make("CartPole-v1", render_mode="human")

### Inhalte der Umgebung

Es gibt einige Begriffe die mit der Umgebung zusammenhängen

Spaces

    Action Space: Beschreibt die Aktionen die der Agent ausführen kann
    Observation Space: Liste der Daten die beim Ausführen von Actions in der Umgebung erzeugt werden -> Training mihilfe dieser Daten
    Discrete, Box, Tuple, Dict, ...

Funktionen der Umgebung

    init: Setzt die Umgebung auf (Setup)
    reset(): Setzt die Umgebung auf den Start zurück
    render(): Zeichne die Umgebung (falls möglich)
    step(): Führe die Action aus die vom Agenten ausgewählt wird
    close(): Beende die Umgebung


In [7]:
durchgaenge = 5
for i in range(durchgaenge): # Teste die Umgebung x mal ab
    state = env.reset() # Die Umgebung für den nächsten Durchlauf zurücksetzen
    done = False
    score = 0

    while not done:
        env.render() # zeichne die Umgebung
        action = env.action_space.sample()  #Nimmt ein Random Element aus dem Action Space (Links oder Rechts)
        state, reward, done, term, info = env.step(action)
        # state: Observation Space, reward: Belohnung, done: Fertig (True/False), term: Termination (vorzeitiges Beenden), info: Extra Infos
        score +=reward
    print(f"Durchgang {i}, Score: {score}")
env.close()


C:\Users\de2\OneDrive - ppedv AG\Anlagen\Schulungsunterlagen\Python\Power-WochePython-VomAnfaengerzumDataScienceProfi-245349\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Durchgang 0, Score: 13.0
Durchgang 1, Score: 50.0
Durchgang 2, Score: 21.0
Durchgang 3, Score: 21.0
Durchgang 4, Score: 10.0


In [9]:
env.action_space.sample() # Discrete(X) Aufsteigende Liste von 0 bis X

np.int64(0)

In [11]:
env.observation_space.sample() # Position, Geschwindigkeit, Winkel der Stange, L-R Geschwindigkeit der Stange


array([-4.7917166 ,  0.44034106, -0.19250216, -1.3786602 ], dtype=float32)


### RL Model trainieren

Für ein RL Modell benötigen wir einen Algorithmus

    PPO: Proximal Policy Optimation
    https://stable-baselines.readthedocs.io/en/master/guide/algos.html

DummyVecEnv (optional)

    Simulation mehrerer Agenten gleichzeitig
    Sehr nützlich um Zeit zu sparen

Policy:

    Bestimmt wie der Algorithmus lernt
    https://stable-baselines.readthedocs.io/en/master/modules/policies.html



In [12]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

In [13]:
env = gym.make("CartPole-v1")
env = DummyVecEnv([lambda: env])
model = PPO("MlpPolicy", env, verbose=1) # MLP: Multilayer Perceptron -> effektiv Dense Layer

Using cpu device


In [14]:
model.learn(total_timesteps=20000)

-----------------------------
| time/              |      |
|    fps             | 996  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 620        |
|    iterations           | 2          |
|    time_elapsed         | 6          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.00917631 |
|    clip_fraction        | 0.101      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.686     |
|    explained_variance   | -0.00733   |
|    learning_rate        | 0.0003     |
|    loss                 | 6.22       |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0154    |
|    value_loss           | 48.5       |
----------------------------------------
-----------------------------------------
| time/   

#### Modell speichern
Wird benötigt, um das Modell auf eine neue Umgebung anzuwenden

In [15]:
model.save("models/CartPole")

#### Modell evaluieren

In [16]:
from stable_baselines3.common.evaluation import evaluate_policy

In [17]:
evaluate_policy(model, env, n_eval_episodes=10)

C:\Users\de2\OneDrive - ppedv AG\Anlagen\Schulungsunterlagen\Python\Power-WochePython-VomAnfaengerzumDataScienceProfi-245349\.venv\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(np.float64(500.0), np.float64(0.0))

In [18]:
env.close()

### Modell testen

In [19]:
import numpy as np


In [20]:
env = gym.make("CartPole-v1")

In [21]:
model = PPO.load("Models/CartPole", env = env)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [25]:
durchgaenge = 5
for i in range(durchgaenge):
    observation = env.reset()
    done = False
    score = 0
    observation = observation[0].reshape(1,4)

    while not done:
        env.render()
        action, _ = model.predict(observation)
        if action.ndim == 1:
            action = action[0]
        observation, reward, done,term, info = env.step(action)
        score += reward
    print(f"Durchgang {i}, Score: {score}")
env.close()


Durchgang 0, Score: 1073.0
Durchgang 1, Score: 413.0
Durchgang 2, Score: 566.0
Durchgang 3, Score: 445.0
Durchgang 4, Score: 342.0
